# Chapter 1 · Introduction to AI Agents

The same loop as the textbook, wired to a real model through Lumen, the University of Illinois campus LLM service. Every step is printed. The job is the one from the chapter: a client wants Deere's revenue growth last quarter compared with Caterpillar, by noon.

**Before you run:**

1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat) with your Illinois account.
2. Open your [profile page](https://lumen.ncsa.illinois.edu/profile), scroll down to **API key**, and create one.
3. In Colab, add it as a secret named `LUMEN_API_KEY` (key icon in the left sidebar) and enable notebook access.

The notebook uses `glm-5.3-flash`. It is the recommended model for this workshop: fast, reliable with tool calls, and free on Lumen. Stick with it unless a section says otherwise.

In [ ]:
%pip install -q openai

In [ ]:
import os
from google.colab import userdata
os.environ["LUMEN_API_KEY"] = userdata.get("LUMEN_API_KEY")

from openai import OpenAI
client = OpenAI(
    base_url="https://lumen.ncsa.illinois.edu/v1",
    api_key=os.environ["LUMEN_API_KEY"],
)
MODEL = "glm-5.3-flash"  # highly preferred for this workshop

## Tools

The functions are tiny and read-only. The schemas are the contract the model sees. Note `strict: True` and `additionalProperties: False`. Lumen speaks the OpenAI chat-completions format, so each tool is wrapped as `{"type": "function", "function": {...}}` with the schema under `parameters`.

In [ ]:
FINANCIALS = {
    ("DE", "Q2-2026"):   {"name": "Deere", "revenue": 13.8e9, "yoy": 0.064},
    ("CAT", "Q2-2026"):  {"name": "Caterpillar", "revenue": 16.9e9, "yoy": 0.031},
    ("NVDA", "Q2-2026"): {"name": "NVIDIA", "revenue": 52.4e9, "yoy": 0.58},
    ("AAPL", "Q2-2026"): {"name": "Apple", "revenue": 96.1e9, "yoy": 0.05},
    ("MSFT", "Q2-2026"): {"name": "Microsoft", "revenue": 76.3e9, "yoy": 0.17},
}
PRICES = {"DE": 512.40, "CAT": 398.15, "NVDA": 181.22, "AAPL": 232.90, "MSFT": 512.06}

def get_financials(ticker, period="Q2-2026"):
    row = FINANCIALS.get((ticker, period))
    return {**row, "ticker": ticker, "period": period} if row else {"error": f"no data for {ticker} {period}"}

TOOLS = {"get_financials": get_financials}

TOOL_SCHEMAS = [
    {"type": "function",
     "function": {
         "name": "get_financials",
         "description": "Quarterly revenue and YoY growth for one ticker. Use only when the user asks about revenue, growth, or earnings; call it once per company when comparing.",
         "strict": True,
         "parameters": {"type": "object",
                        "properties": {"ticker": {"type": "string", "enum": list(PRICES)},
                                       "period": {"type": "string", "enum": ["Q2-2026"]}},
                        "required": ["ticker", "period"], "additionalProperties": False}}},
]

## The loop

Identical in shape to the textbook. The only differences are the message format the API expects (an assistant message carrying `tool_calls`, answered by one `tool` message per call) and that a reply can contain several tool calls at once, which we run and return together.

In [ ]:
import json, time

def _describe(name, args):
    """Plain-English description of a tool call, not a JSON blob — this book is for non-technical readers."""
    detail = ", ".join(str(v) for v in args.values())
    return f"{name.replace('_', ' ')}" + (f" ({detail})" if detail else "")

def _describe_result(result):
    """What a tool handed back, in words instead of a dict."""
    if isinstance(result, dict) and "error" in result:
        return f"nothing found — {result['error']}"
    if isinstance(result, dict):
        return ", ".join(f"{k.replace('_', ' ')} {v}" for k, v in result.items())
    return str(result)

def agent(question, tools=TOOLS, schemas=TOOL_SCHEMAS, system=None, max_steps=6):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    log = []
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, tools=schemas, messages=messages)
        msg = response.choices[0].message
        tool_calls = msg.tool_calls or []
        if not tool_calls:
            text = (msg.content or "").strip()
            log.append({"step": step, "kind": "text", "text": text})
            print(f"Step {step + 1}: answered — {text}")
            return text, log

        messages.append(msg)
        for tc in tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            t0 = time.time()
            try:
                result = tools[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            ms = round((time.time() - t0) * 1000, 2)
            log.append({"step": step, "kind": "tool_call", "tool": name, "args": args, "result": result, "ms": ms})
            print(f"Step {step + 1}: looked up {_describe(name, args)} -> {_describe_result(result)}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    log.append({"step": max_steps, "kind": "budget_exhausted"})
    return "Stopped: step budget exhausted.", log

answer, log = agent("Compare Deere's revenue growth with Caterpillar's last quarter")

## A live business example: the client comparison, by noon

The same loop, now doing the job Priya's team does thirty times a week. One read-only tool, `get_financials`. Deliberately no `send_to_client` tool: the model drafts, the analyst sends.

The system prompt asks for the memo shape the firm wants: both figures, the gap, and a source tag, because the handbook says every figure that leaves the building must cite its source.

In [ ]:
SYSTEM = ("You are a research assistant at Champaign Capital Research, an equity research firm. "
          "When asked to compare companies, look up each one with get_financials, then write a two-sentence client memo: "
          "both figures (revenue and YoY growth), who is growing faster and by roughly how much, and a source tag like "
          "[source: Q2-2026 filings via get_financials]. Never state a figure you did not get from a tool. "
          "You cannot send anything to clients; an analyst does.")

for pair in ["Deere and Caterpillar", "NVIDIA and Apple", "Microsoft and Apple"]:
    answer, log = agent(f"Compare revenue growth for {pair} last quarter", system=SYSTEM)
    print()

Compare with the textbook's mock run. Did the real model look up both companies before writing? Did it ask for them in one reply or one at a time? Did every memo carry a `[source: …]` tag? If not, the fix belongs in the tool description or the system prompt, and you should be able to say which.

Now the case the mock handles with an error. The schema's `ticker` enum does not contain `TSLA`, so with `strict: True` the model cannot even ask for it:

In [ ]:
answer, log = agent("Compare Deere's revenue growth with Tesla's last quarter", system=SYSTEM)

## Exercise

1. Run the comparison above and compare its tool sequence and memo with the textbook's mock run.
2. Add `get_price(ticker)` to `TOOLS` and a closed schema for it to `TOOL_SCHEMAS`, then run the question below. Expect four tool calls and one memo.
3. Read what the model did with Tesla. Write two sentences: what it said, and whether a client could tell the firm holds no data.
4. Add the total elapsed time of the whole run to the final log entry.
5. Write three sentences: one tool call the model made that you would not have made, and the schema change that would prevent it.

In [ ]:
# 2. your get_price tool and schema here


answer, log = agent("Is Deere's revenue growth better than Caterpillar's, and what are both trading at?", system=SYSTEM)
print()
for entry in log:
    if entry["kind"] == "tool_call":
        print(f"Step {entry['step'] + 1}: looked up {_describe(entry['tool'], entry['args'])} ({entry['ms']} ms)")
    else:
        print(f"Step {entry['step'] + 1}: {entry.get('text', 'stopped')}")

## Optional: compare models

Lumen hosts more than one model; the list is in the model picker at [lumen.ncsa.illinois.edu/chat](https://lumen.ncsa.illinois.edu/chat). Set `MODEL` to a different one and rerun the exercise question. Count the tool calls and read the final answer. Which one would you ship, and why? (For everything else in this workshop, stay on `glm-5.3-flash`.)